## Environment Setup

In [ ]:
# 1. Uninstall the conflicting packages
pip uninstall -y bitsandbytes torch torchvision torchaudio nvidia-nvjitlink-cu13

# 2. Explicitly install PyTorch compiled for CUDA 12.4
pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124

# 3. Reinstall bitsandbytes (it will now correctly detect and build against CUDA 12)
pip install bitsandbytes

In [ ]:
!pip install -U torch torchvision torchaudio
!pip install -U transformers accelerate peft bitsandbytes datasets qwen-vl-utils sentence-transformers json-repair

## Imports & Global Configurations

In [1]:
import os
import re
import json
import copy
import torch
import numpy as np
from PIL import Image
from transformers import AutoProcessor, AutoModelForImageTextToText
from peft import PeftModel
from qwen_vl_utils import process_vision_info
from sentence_transformers import SentenceTransformer
import json_repair

# Force Hugging Face to download massive weights to the persistent volume
os.environ["HF_HOME"] = "/workspace/huggingface_cache"

# --- Global Configurations ---
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SAFE_DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float32

QWEN_BASE_ID = "Qwen/Qwen2.5-VL-7B-Instruct"
EXTRACTOR_LORA_PATH = "/workspace/ImageToSpec_Stage1/qwen_lora_final"
VERIFIER_LORA_PATH = "/workspace/Stage2_ChartCheck/qwen_lora_verifier/checkpoint-900"

## Load Models (Multi-Adapter Setup)

In [2]:
print("1. Loading Qwen Base Model & Processor...")
qwen_processor = AutoProcessor.from_pretrained(QWEN_BASE_ID)
base_model = AutoModelForImageTextToText.from_pretrained(
    QWEN_BASE_ID, 
    device_map="auto", 
    torch_dtype=SAFE_DTYPE, 
    attn_implementation="sdpa"
)

print("2. Loading LoRA Adapters (Extractor & Verifier)...")
# Load the first adapter and name it
qwen_model = PeftModel.from_pretrained(base_model, EXTRACTOR_LORA_PATH, adapter_name="extractor")
# Load the second adapter alongside it
qwen_model.load_adapter(VERIFIER_LORA_PATH, adapter_name="verifier")
qwen_model.eval()

print("3. Loading Semantic Retriever...")
semantic_retriever = SentenceTransformer("all-MiniLM-L6-v2")

print("✅ All models loaded successfully into VRAM!")

1. Loading Qwen Base Model & Processor...


preprocessor_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

chat_template.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ All models loaded successfully into VRAM!


## Data Pruning & Retrieval Logic

In [3]:
def retrieve_top_k_series(claim, spec, k=3):
    series_list = spec.get("ser", [])
    if not isinstance(series_list, list) or len(series_list) <= k: return spec
    docs = [claim] + [json.dumps(s) for s in series_list]
    with torch.no_grad():
        embs = semantic_retriever.encode(docs, convert_to_tensor=True)
        sims = torch.nn.functional.cosine_similarity(embs[0:1], embs[1:])
    top_k = sims.topk(k).indices.sort().values.tolist()
    spec["ser"] = [series_list[i] for i in top_k]
    spec["is_truncated"] = True
    return spec

def lightweight_prune_for_llm(raw_spec_dict, claim_text):
    """Surgically prunes the spec to preserve semantics and math, while dropping visual noise."""
    if not isinstance(raw_spec_dict, dict): return {}
    cleaned = copy.deepcopy(raw_spec_dict)
    
    if 'ser' not in cleaned or not isinstance(cleaned['ser'], list) or len(cleaned['ser']) == 0: 
        return cleaned

    def compress_val(v):
        if isinstance(v, float): return int(v) if v.is_integer() else round(v, 3)
        return v

    # 1. Strip Root-Level Visual/System Noise
    cleaned.pop('legend', None)
    cleaned.pop('tooltip', None)

    # 2. Apply Semantic Retriever 
    cleaned = retrieve_top_k_series(claim_text, cleaned, k=3)

    # 3. Clean Remaining Series
    topo = cleaned.get('topo', {})
    topo_type = str(topo.get('type', '')).strip().lower() if isinstance(topo, dict) else ''
    
    for ser in cleaned.get('ser', []):
        if isinstance(ser, dict):
            # Aggressively drop visual flags
            for key in ['ds', 'is_subsampled', 'critical_points_retained', 'color', 'uncertainty_metric']:
                ser.pop(key, None)
            
            # Pie charts do not mathematically possess trends
            if 'pie' in topo_type: 
                ser.pop('trend', None)
                ser.pop('stats', None)
            
            # Compress float values in data arrays
            if 'data' in ser and isinstance(ser['data'], list):
                compressed_data = []
                for pt in ser['data']:
                    if isinstance(pt, list):
                        compressed_data.append([compress_val(v) for v in pt])
                    else:
                        compressed_data.append(compress_val(pt))
                ser['data'] = compressed_data

    return cleaned

## Simulation Samples

In [4]:
simulation_samples = [
     # ── IMAGE 2: Fox Business Bush Tax Cuts Bar Chart ──────────────────────
    {
        "image_path": "bush_tax.png",
        "title_description": "Bar chart showing top tax rate comparison: current rate of 35% vs. projected rate of 39.6% if Bush tax cuts expire on Jan. 1, 2013",
        "chart_type": "bar_chart",
        "claim": "The top tax rate would increase if the Bush tax cuts expire on January 1, 2013.",
        "truth": "(SUPPORTED claim)"
        # Rationale: The chart clearly shows 35% → 39.6%, a factual increase.
        # Any viewer would correctly read this direction of change.
    },
    {
        "image_path": "bush_tax.png",
        "title_description": "Bar chart showing top tax rate comparison: current rate of 35% vs. projected rate of 39.6% if Bush tax cuts expire on Jan. 1, 2013",
        "chart_type": "bar_chart",
        "claim": "The top tax rate would nearly double if the Bush tax cuts expire.",
        "truth": "(REFUTED claim)"
        # Common mistake: The y-axis starts at 34% instead of 0%, making the
        # Jan 2013 bar look roughly 5-6x taller than the 'NOW' bar visually.
        # A person relying on bar HEIGHT rather than the labeled values (35%
        # vs 39.6%) would dramatically overestimate the magnitude of increase
        # — a classic truncated-axis hallucination. The actual increase is
        # only ~4.6 percentage points (~13% relative increase), not a doubling.
    },
]

In [ ]:
ood_simulation_samples = [
    {
    "image_path": "27201.jpeg",
    "title_description": "Where the Super Rich Reside: Number of billionaires by country/territory of citizenship (2026)",
    "chart_type": "choropleth_map",
    "claim": "The United States has more billionaires than China and India combined.",
    "truth": " (SUPPORTED claim)"
},
{
    "image_path": "27201.jpeg",
    "title_description": "Where the Super Rich Reside: Number of billionaires by country/territory of citizenship (2026)",
    "chart_type": "choropleth_map",
    "claim": "Germany is home to more billionaires than India.",
    "truth": " (REFUTED claim)"
},
]

## E2E Execution Loop (VLM Rationale Generation)

In [5]:
output_json_path = "results_generative_box_heat_plots.json"
simulation_results = []

print("🚀 Starting Continuous Generative Verification Pipeline...\n" + "="*60)

for idx, sample in enumerate(simulation_samples):
    print(f"\n--- Processing Sample {idx + 1} ---")
    if not os.path.exists(sample["image_path"]):
        print(f"⚠️  Image not found: {sample['image_path']}. Skipping.")
        continue

    # ==========================================
    # PHASE 1: EXTRACT SPEC FROM IMAGE
    # ==========================================
    qwen_model.set_adapter("extractor")
    
    ext_messages = [
        {"role": "system", "content": "You output strictly valid JSON with no markdown formatting."},
        {"role": "user", "content": [
            {"type": "image", "image": sample["image_path"], "max_pixels": 768 * 768},
            {"type": "text", "text": "Extract the Extended ChartSpec JSON from this chart image."}
        ]}
    ]
    ext_text = qwen_processor.apply_chat_template(ext_messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(ext_messages)
    ext_inputs = qwen_processor(text=[ext_text], images=image_inputs, videos=video_inputs, padding=True, return_tensors="pt").to(DEVICE)

    with torch.no_grad():
        ext_ids = qwen_model.generate(**ext_inputs, max_new_tokens=2048, do_sample=False, repetition_penalty=1.1)
    
    ext_trimmed = ext_ids[0][len(ext_inputs.input_ids[0]):]
    raw_text = qwen_processor.decode(ext_trimmed, skip_special_tokens=True)
    
    try:
        raw_spec = json_repair.repair_json(re.sub(r'\}\]\}\],\"rel\"', '}],\"rel\"', raw_text), return_objects=True)
    except Exception:
        raw_spec = {}

    # ==========================================
    # PHASE 2: PRUNE & FORMAT
    # ==========================================
    cleaned_spec = lightweight_prune_for_llm(raw_spec, sample["claim"])
    cleaned_spec_str = json.dumps(cleaned_spec, separators=(',', ':'))

    # ==========================================
    # PHASE 3: REASONING & VERDICT
    # ==========================================
    qwen_model.set_adapter("verifier")
    
    ver_sys_prompt = "You are an expert chart fact-checker. Analyze the JSON specification to verify the user's claim."
    ver_user_prompt = f"Verify this claim.\nClaim: {sample['claim']}\nChart Specification: {cleaned_spec_str}"
    
    ver_messages = [
        {"role": "system", "content": ver_sys_prompt}, 
        {"role": "user", "content": ver_user_prompt}
    ]
    
    ver_text = qwen_processor.apply_chat_template(ver_messages, tokenize=False, add_generation_prompt=True)
    ver_inputs = qwen_processor(text=[ver_text], padding=True, return_tensors="pt").to(DEVICE)
    
    with torch.no_grad():
        ver_ids = qwen_model.generate(
            **ver_inputs, 
            max_new_tokens=150, 
            do_sample=False,              
            repetition_penalty=1.05,      
            pad_token_id=qwen_processor.tokenizer.pad_token_id
        )
        
    ver_trimmed = ver_ids[0][len(ver_inputs.input_ids[0]):]
    output_text = qwen_processor.decode(ver_trimmed, skip_special_tokens=True).strip()

    # Parse Output
    pred_val_str = "REFUTED"
    match = re.search(r"Verdict:\s*(SUPPORTED|REFUTED)", output_text, re.IGNORECASE)
    if match:
        pred_val_str = match.group(1).upper()
    elif "supported" in output_text.lower(): 
        pred_val_str = "SUPPORTED"
        
    pred_rationale = output_text
    if match:
        pred_rationale = output_text[:match.start()].replace("Rationale:", "").strip()

    # Console output
    print(f"Claim                : {sample['claim']}")
    print(f"Truth                : {sample['truth'].strip()}")
    print(f"Qwen Prediction      : {pred_val_str}")
    print(f"Qwen Rationale       : {pred_rationale}")

    # Incremental save
    simulation_results.append({
        "sample_id":         idx + 1,
        "image_path":        sample["image_path"],
        "chart_type":        sample["chart_type"],
        "claim":             sample["claim"],
        "truth":             sample["truth"].strip(),
        "prediction":        pred_val_str,
        "rationale":         pred_rationale,
        "pruned_spec":       cleaned_spec,
    })

    with open(output_json_path, "w", encoding="utf-8") as f:
        json.dump(simulation_results, f, indent=4, ensure_ascii=False)

print("\n" + "="*60 + "\n✅ Simulation Complete!")

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


🚀 Starting Continuous Generative Verification Pipeline...

--- Processing Sample 1 ---
Claim                : In Experiment 18, Data Set 2 has a higher median value than Data Set 1.
Truth                : (SUPPORTED claim)
Qwen Prediction      : SUPPORTED
Qwen Rationale       : The chart shows that in Experiment 18, Data Set 2 has a higher median value than Data Set 1, which is indicated by the higher point on the graph for Data Set 2 compared to Data Set 1.

--- Processing Sample 2 ---
Claim                : Across most experiments shown, (blue) consistently displays a larger number of lower-end outliers compared to Data Set 2.
Truth                : (SUPPORTED claim)
Qwen Prediction      : SUPPORTED
Qwen Rationale       : The blue line is higher than the red line across all experiments.

--- Processing Sample 3 ---
Claim                : In Experiment 24, Data Set 2 shows a significantly higher maximum value, including outliers, than Data Set 1.
Truth                : (REFUTED claim)